# Stage 2: train

**Goal.** Train the U-Net of Wagner et al. (2024) on the patch cache with the paper's recipe, and
keep the best checkpoint.

**Command-line equivalent:** `p037 train --config <config>` (on BluePebble: `sbatch train.sbatch <config>`)

**Code:** `src/p037/model.py`, `src/p037/data.py`, `src/p037/train.py`

Experiments that use the published weights (`init: pretrained`) skip this stage.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from p037 import plots                      # sets the figure style used in the report
from p037.config import REPO_ROOT, load_config
from p037.datasets import get_dataset

# The experiment this notebook works on. smoke.yaml is generated data: it runs anywhere in minutes
# and its numbers mean nothing. On BluePebble use analysis/p037/configs/exp_b_usf.yaml.
# (Setting the environment variable P037_CONFIG before starting Jupyter does the same for all notebooks.)
CONFIG = os.environ.get("P037_CONFIG", "analysis/p037/configs/smoke.yaml")
cfg = load_config(REPO_ROOT / CONFIG)
ds = get_dataset(cfg)
DRY_RUN = cfg.dataset == "synthetic"
print(f"experiment {cfg.name!r} on dataset {cfg.dataset!r}")
print("dataset files:", cfg.dataset_dir)
print("run folder:   ", cfg.out)

## 2.1 The model

`s1water.model.UNetS1Water`: a U-Net with 4 input channels and one output, the probability per
pixel. It applies the sigmoid itself, so its output is already a probability.

`init: random` builds it with fresh weights (Experiment B). `init: pretrained` would load the
published Amazon water weights instead.

In [ ]:
import torch

from p037.model import build_model, pick_device

device = pick_device()
torch.manual_seed(cfg.seed)
model = build_model(cfg, init="random").to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {device} | parameters: {n_params / 1e6:.1f} M | dropout: {cfg.dropout}")

## 2.2 One batch

Each item is three arrays: the image (8-bit, the four channels named in the config), the target
(1 where the label is one of `target_classes`) and a mask (0 where the label is invalid).
The image is divided by 255 on the device, as in the original code.

In [ ]:
from p037.data import make_loaders

train_loader, val_loader, train_rows, val_rows = make_loaders(cfg, device)
x8, y, m = next(iter(train_loader))
print("image", tuple(x8.shape), x8.dtype, "| target", tuple(y.shape), "| mask", tuple(m.shape))
print(f"flood share of this batch: {100 * float((y * m).sum() / m.sum()):.2f} % of valid pixels")
x = x8.to(device).float() / 255
y, m = y.to(device), m.to(device)

## 2.3 The loss

Binary cross-entropy plus Dice, as in the paper (section 2.6), with two details that matter here:

- **Masked.** Invalid pixels contribute nothing.
- **Dice over the whole batch.** Most patches contain no flood, and the Dice score of an empty
  patch is decided by its smoothing constant alone, so per-patch Dice would mostly be noise.

An untrained model should give a cross-entropy near ln 2 = 0.69 (it knows nothing) plus a Dice
term between about 0.7 and 1 (little overlap yet; closer to 1 the rarer flood is in the batch).
A first loss of roughly 1.3 to 1.7 is therefore the expected starting point. A value far outside
that range before any training points to a problem in the data or the mask.

In [ ]:
from p037.train import bce_dice_loss

model.train()
with torch.no_grad():
    print(f"loss of the untrained model on this batch: {float(bce_dice_loss(model(x), y, m)):.3f}")

## 2.4 Sanity check: can it memorise one batch?

Before spending GPU hours, check that the model, loss and optimiser are wired correctly: trained
repeatedly on a single batch, the loss must fall steadily. If it does not, something is broken
(labels misaligned, mask inverted, learning rate), and a full run would only hide it for longer.

In [ ]:
STEPS = 40 if device == "cpu" else 200
opt = torch.optim.Adam(model.parameters(), lr=cfg.lr * (10 if DRY_RUN else 1))
history = []
for step in range(STEPS):
    loss = bce_dice_loss(model(x), y, m)
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(float(loss.detach()))

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(history, color=plots.BLUE)
ax.set_xlabel("step on the same batch"); ax.set_ylabel("loss"); ax.set_title("Overfitting one batch", loc="left")
print(f"loss: {history[0]:.3f} -> {history[-1]:.3f}")

## 2.5 Full training

`train(cfg)` runs the real thing: every epoch it trains on all training patches, scores the
validation patches, appends a row to `log.csv` and writes `last.pt`. It keeps `best_f1.pt` and
`best_accuracy.pt`. Run again, it resumes from `last.pt`, and an epoch is the same whether the
run reached it in one job or after a resume.

On BluePebble submit it as a job. The cell below trains here only for the dry run.

In [ ]:
from p037.train import train

if DRY_RUN:
    state = train(cfg, device)
    print(state)
else:
    print("Submit on BluePebble:  sbatch analysis/p037/slurm/train.sbatch", CONFIG)

## 2.6 Reading the training log

Two things to look for:

- **Loss.** Training and validation loss should both fall. Validation rising while training keeps
  falling is overfitting, and it is when early stopping (`patience`) ends the run.
- **F1 against accuracy.** Accuracy is high from the first epoch because most pixels are not
  flood. F1 is the curve that shows whether flood is being found.

In [ ]:
log_path = cfg.out / "log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
    a.plot(log["epoch"], log["train_loss"], color=plots.MUTED, label="training")
    a.plot(log["epoch"], log["val_loss"], color=plots.BLUE, label="validation")
    a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.set_title("Loss", loc="left")
    b.plot(log["epoch"], log["val_f1"], color=plots.BLUE, label="validation F1")
    b.plot(log["epoch"], log["val_accuracy"], color=plots.ORANGE, label="validation accuracy")
    best = log.loc[log["val_f1"].idxmax()] if log["val_f1"].notna().any() else None
    if best is not None:
        b.axvline(best["epoch"], color=plots.AXIS, lw=1)
    b.set_xlabel("epoch"); b.set_ylim(0, 1.02); b.legend(); b.set_title("Validation scores", loc="left")
    fig.tight_layout()
    display(log.tail(5))
    print(f"seconds per epoch (median): {log['seconds'].median():.0f}")
else:
    print("no log yet:", log_path)

## Check before moving on

- One batch can be memorised (section 2.4).
- Validation F1 rises above zero within the first epochs of the real run. If it stays at zero
  while accuracy is high, the model has collapsed to "no flood anywhere".
- Note which validation set the F1 refers to. With the official split it comes from patches next
  to training patches, so it is an upper bound, not a measure of how the model does elsewhere.

Next: `03_evaluate.ipynb`.